# 🏥 ViBioMIR Interactive Search & Offline Testing (Local & Kaggle)
Notebook này được thiết kế để chạy mượt mà cả trên **máy Local** (với môi trường `.venv`) lẫn trên **Kaggle / Colab**:
1. **Tự động cấu hình Working Directory** về gốc dự án (`med-doc-retrieval`).
2. **Kiểm tra Index sẵn có** (FAISS SQ8, BM25s, SQLite Metadata) mà không cần tải lại nếu đã có trên máy.
3. **Thử nghiệm tìm kiếm tương tác (Interactive Query Search)**: Tra cứu nhanh bất kỳ câu hỏi lâm sàng nào.
4. **Kiểm tra Top 1 F2-Macro Strategy**: Trích xuất gọn 4 docs x 1 chunk để tối ưu điểm $F_2$.
5. **Chạy trích xuất toàn bộ 1,200 câu hỏi** và xuất file `submission_top1_strategy.zip`.

In [ ]:
# 1. Kiểm tra môi trường & Thư viện
try:
    import faiss
    import pyvi
    import bm25s
    import flagembedding
    from sentence_transformers import SentenceTransformer
    print("✅ Môi trường hiện tại đã cài đặt đầy đủ tất cả thư viện cần thiết!")
except ImportError as e:
    print(f"⚠️ Thiếu thư viện ({e}). Đang tiến hành cài đặt...")
    !pip install -q uv
    !uv pip install --system --no-cache faiss-gpu pyarrow "huggingface_hub[cli]" bm25s rank-bm25 pyvi jieba loguru pydantic pyyaml tqdm flagembedding sentence-transformers

In [ ]:
# 2. Cấu hình Thư mục Gốc Dự án (Project Root) và Python Path
import os
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()
if (current_dir / "src").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "src").exists():
    PROJECT_ROOT = current_dir.parent
elif Path("/kaggle/working/med-doc-retrieval").exists():
    PROJECT_ROOT = Path("/kaggle/working/med-doc-retrieval")
else:
    PROJECT_ROOT = current_dir

# Chuyển Working Directory về thư mục gốc của dự án
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("📁 Current Working Directory:", os.getcwd())
print("🐍 Python Executable:", sys.executable)

In [ ]:
# 3. Kiểm tra hoặc Tải Index (Tự động nhận diện index trên máy local)
from pathlib import Path
from huggingface_hub import HfApi

indices_dir = Path("data/indices")
indices_dir.mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)

# Danh sách file cần thiết cho Hybrid Retrieval (FAISS SQ8 + SQLite + BM25s)
sync_files = [
    ("indices/dense_index_sq8.faiss", "data/indices/dense_index_sq8.faiss"),
    ("indices/chunks_meta_slim.sqlite", "data/indices/chunks_meta_slim.sqlite"),
    ("indices/dense_metadata_sq8.json", "data/indices/dense_metadata_sq8.json"),
    ("indices/bm25s_index/data.csc.index.npy", "data/indices/bm25s_index/data.csc.index.npy"),
    ("indices/bm25s_metadata.json", "data/indices/bm25s_metadata.json"),
    ("processed/queries_enriched.jsonl", "data/processed/queries_enriched.jsonl"),
]

missing = [(rem, loc) for rem, loc in sync_files if not Path(loc).exists() or Path(loc).stat().st_size == 0]

if missing:
    print(f"📥 Phát hiện {len(missing)} file chưa có, bắt đầu đồng bộ từ Hugging Face Bucket 'nieths/ViBioMIR'...")
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    api = HfApi(token=HF_TOKEN or None)
    api.download_bucket_files(bucket_id="nieths/ViBioMIR", files=missing, token=HF_TOKEN or None)
    print("✅ Hoàn tất tải file index!")
else:
    print("✅ Toàn bộ file index (SQ8 FAISS + BM25s + SQLite Slim) đã có sẵn trên máy!")

# Liệt kê các file trong thư mục index
for p in sorted(indices_dir.iterdir()):
    if p.is_file():
        print(f"  • {p.name:<30} ({p.stat().st_size / (1024*1024):.2f} MB)")
    elif p.is_dir():
        print(f"  📁 {p.name}/")

In [ ]:
# 4. Khởi tạo Pipeline & Load Index (Hybrid: FAISS SQ8 + BM25s + SQLite)
from src.config import load_config
from src.pipeline import MedicalRetrievalPipeline

config = load_config("configs/config.yaml")
config.pubmed.source_api = "none"  # Tắt HTTP API ngoài, dùng cache local để inference nhanh nhất

pipeline = MedicalRetrievalPipeline(config=config)
pipeline.load_indices(
    indices_dir="data/indices",
    dense_dir="data/indices",
    sparse_dir="data/indices",
    chunks_meta_path="data/indices/chunks_meta_slim.sqlite",
)
print("🚀 Pipeline Hybrid (FAISS SQ8 + BM25s) đã khởi tạo thành công!")

In [ ]:
# 5. Tìm kiếm Tương tác Thử nghiệm (Interactive Search)
test_query = "Bệnh nhân bị xuất huyết giảm tiểu cầu vô căn điều trị bằng thuốc gì đầu tay?"

print(f"🔍 Đang tìm kiếm cho câu hỏi: '{test_query}'")
result = pipeline.search_query(test_query)

print("\n--- 🏆 TOP TÀI LIỆU (DOC_IDS) ---")
for rank, did in enumerate(result["relevant_docs"][:5], 1):
    print(f"  #{rank} Doc ID: {did}")

print("\n--- 📑 TOP CHUNKS BẰNG CHỨNG TRÍCH XUẤT ---")
for rank, ch in enumerate(result["relevant_chunks"][:3], 1):
    snippet = ch['chunk_text'][:250].replace('\n', ' ')
    print(f"  #{rank} Doc ID {ch['doc_id']}:\n     \"{snippet}...\"\n")

In [ ]:
# 6. Chạy thử nghiệm trên 5 query trong tập test
import json
import pyarrow.parquet as pq

queries_path = Path("data/processed/queries_enriched.jsonl")
sample_queries = []

if queries_path.exists():
    with open(queries_path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 5: break
            sample_queries.append(json.loads(line))
elif Path("data/raw/vibio_mir/query.parquet").exists():
    table = pq.read_table("data/raw/vibio_mir/query.parquet")
    sample_queries = table.to_pylist()[:5]

print(f"Chạy kiểm tra trên {len(sample_queries)} câu hỏi ngẫu nhiên:")
for q in sample_queries:
    qid = q["id"]
    q_text = q.get("query") or q.get("original_query")
    res = pipeline.search_query(q_text)
    print(f"\n[Query #{qid}]: {q_text[:70]}...")
    print(f"  -> Top Docs: {res['relevant_docs'][:4]}")
    print(f"  -> Số Chunks: {len(res['relevant_chunks'])}")

In [ ]:
# 7. Chạy Inference toàn bộ tập test với CHIẾN LƯỢC TOP 1 (Tối ưu Macro F2)
# Lưu ý: Áp dụng top_k_docs=4, top_k_chunks=4, max_chunks_per_doc=1 để tránh bị phạt rác ở Chunk Precision
import sys

!{sys.executable} scripts/run_inference_only.py \
    --indices-dir "data/indices" \
    --dense-dir "data/indices" \
    --sparse-dir "data/indices" \
    --chunks-meta-path "data/indices/chunks_meta_slim.sqlite" \
    --hybrid-top-k 200 \
    --top-k-docs 4 \
    --top-k-chunks 4 \
    --max-chunks-per-doc 1 \
    --batch-size 16 \
    --disable-pubmed-network \
    --output-name submission_top1_strategy.json

In [ ]:
# 8. Kiểm tra File Submission đã sinh
import base64
from IPython.display import HTML, display

sub_dirs = [Path("outputs/submissions"), Path("/kaggle/working"), Path(".")]
zip_candidates = []
for d in sub_dirs:
    if d.exists():
        zip_candidates.extend(list(d.glob("*.zip")))

if zip_candidates:
    latest_zip = sorted(zip_candidates, key=lambda x: x.stat().st_mtime, reverse=True)[0]
    size_mb = latest_zip.stat().st_size / (1024 * 1024)
    print(f"🎉 File submission mới nhất: {latest_zip.resolve()} ({size_mb:.2f} MB)")
    
    with open(latest_zip, "rb") as f:
        data_b64 = base64.b64encode(f.read()).decode()
    
    html_button = f'''
    <div style="padding:15px; background:#e8f4fd; border:2px solid #2196f3; border-radius:8px; margin-top:10px;">
       <h4 style="color:#0b5394; margin:0 0 8px 0;">📦 Tải Submission ZIP ({latest_zip.name})</h4>
       <a href="data:application/zip;base64,{data_b64}" download="{latest_zip.name}" 
          style="display:inline-block; padding:10px 20px; background:#2196f3; color:white; 
                 text-decoration:none; border-radius:6px; font-weight:bold; font-size:14px;">
          ⬇ BẤM VÀO ĐÂY ĐỂ TẢI VỀ MÁY
       </a>
    </div>
    '''
    display(HTML(html_button))
else:
    print("Chưa tìm thấy file submission .zip nào. Hãy chạy cell 7 trước!")